In [53]:
from sklearn.preprocessing import MinMaxScaler
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM as lstm
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import EarlyStopping
from keras import Input
import pmdarima as pm
from sklearn.metrics import mean_squared_error, mean_absolute_error
import pandas as pd
import numpy as np
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from statsmodels.tsa.seasonal import STL
from sklearn.linear_model import LinearRegression
from datasets import load_dataset

# Modèles

## Persistence

In [54]:
def persistence(train, test=False): # pas d'ensemble de validation
    prediction = train.iloc[-1]
    if type(test) != bool:
        # Évaluation du modèle
        n_test = len(test)
        predictions = prediction * np.ones(n_test)
        rmse = np.sqrt(mean_squared_error(test, predictions))
        mae = mean_absolute_error(test, predictions)
        return prediction, rmse, mae
    return prediction

## ARIMA

In [71]:
def ARIMA(train, saisonalite, test=False, type_produit=False): # pas d'ensemble de validation
    if type_produit:
        min_train = train.min()
        train = train - train.min() + 1  # décale tout pour que min > 0
        train = np.log(train)
    # fit
    fit_arima = pm.auto_arima(train, seasonal=saisonalite)

    if type_produit:
        train = np.exp(train) + min_train - 1 # on remet le train comme avant
    
    if type(test) != bool:
        # Évaluation du modèle
        n_test = len(test)
        predictions = fit_arima.predict(n_periods=n_test)

        # retour à l'échelle originale
        if type_produit:
            predictions = np.exp(predictions) + min_train - 1
        
        rmse_lstm = np.sqrt(mean_squared_error(test, predictions))
        mae_lstm = mean_absolute_error(test, predictions)
        
        return fit_arima, rmse_lstm, mae_lstm
    return fit_arima

## STL + ARIMA sur le résidu

In [72]:
def STL_ARIMA(train, saisonalite, s_window, test=False, type_produit=False): # s_window : largeur de la fenetre saisonniere lors du lissage
    # local pour déterminer la saisonalité
    
    # Transformation log si multiplicatif
    if type_produit:
        min_train = train.min()
        train = train - train.min() + 1  # décale tout pour que min > 0
        train = np.log(train)
    
    # décomposition STL
    stl = STL(train, period=saisonalite, seasonal=s_window)
    
    if type_produit:
        train = np.exp(train) + min_train - 1 # on remet le train comme avant
    
    stl_result = stl.fit() # trouve la décomposition tendance, saisonalité, et bruit
    
    trend = stl_result.trend
    seasonal = stl_result.seasonal
    resid = stl_result.resid

    # extrapolation de la tendance
    t = np.arange(len(trend)).reshape(-1,1)
    model_trend = LinearRegression().fit(t, trend)

    # saisonnalité
    season_pattern = seasonal.iloc[-saisonalite:].values
    
    # fit ARIMA sur les résidus
    fit_resid = pm.auto_arima(resid, seasonal=False)
    
    if type(test) != bool:
        n_test = len(test)

        # Prévision des résidus
        pred_resid = fit_resid.predict(n_periods=n_test)
    
        # tendance future
        t_future = np.arange(len(trend), len(trend)+n_test).reshape(-1,1)
        trend_future = model_trend.predict(t_future)
    
        # répétition de la saisonnalité 
        season_future = np.tile(season_pattern, int(np.ceil(n_test/saisonalite)))[:n_test] # tile répète le motif saisonnier
    
        # reconstruction
        y_pred = trend_future + season_future + pred_resid
    
        # Retour échelle originale
        if type_produit:
            y_pred = np.exp(y_pred)
    
        rmse = np.sqrt(mean_squared_error(test, y_pred))
        mae = mean_absolute_error(test, y_pred)
        
        return model_trend, season_pattern, fit_resid, rmse, mae
    
    return model_trend, season_pattern, fit_resid

## SARIMAX

In [125]:
def SARIMAX_model(y_train, X_train, saisonalite, test_y=False, test_X=False, type_produit=False):
    
    if type_produit:
        min_train = y_train.min()
        y_train = y_train - min_train + 1  # décale tout pour que min > 0
        y_train = np.log(y_train)

    # Entraînement du modèle SARIMAX
    fit_sarimax = pm.auto_arima(
        y=y_train,
        X=X_train,       # la variable cible ne dépend pas seulement de son passé, mais aussi d’autres signaux mesurés par le procédé
        seasonal=saisonalite,
        suppress_warnings=True,   # Cela cache certains warnings produits pendant la recherche du meilleur modèle
        error_action="ignore",    # Si une combinaison de paramètres provoque une erreur, elle est ignorée
        stepwise=True             # Le modèle cherche à trouver de bons paramètres plus intelligemment et plus rapidement 
    )

    if type_produit:
        y_train = np.exp(y_train) + min_train - 1 # on remet le train comme avant
    
    if type(test) != bool:
        # Prédictions
        n_test = len(test_y)   # On veut prédire exactement autant de périodes que la taille du test
        predictions = fit_sarimax.predict(
            n_periods=n_test,        # nombre de périodes futures à prédire.
            X=test_X      # Comme le modèle a été entraîné avec des variables exogènes, il a besoin des valeurs de ces variables sur la période future
        )
    
        # Retour à l'échelle originale si log appliqué
        if type_produit:
            predictions = np.exp(predictions)
    
        # Évaluation
        rmse = np.sqrt(mean_squared_error(test_y, predictions))
        mae = mean_absolute_error(test_y, predictions)
        return fit_sarimax, rmse, mae

    return fit_sarimax

## STL + ARIMAX

In [259]:
def SARIMAX_STL_model(y_train, X_train, saisonalite, s_window, test_y=False, test_X=False, type_produit=False):
    
    # log si multiplicatif
    if type_produit:
        min_train = y_train.min()
        y_train = y_train - min_train + 1
        y_train = np.log(y_train)
    
    # Décomposition STL si demandé
    stl = STL(train, period=saisonalite, seasonal=s_window)
    stl_result = stl.fit()
    trend = stl_result.trend
    seasonal = stl_result.seasonal
    resid = stl_result.resid

    if type_produit:
        y_train = np.exp(y_train) + min_train - 1 # on remet le train comme avant
    
    # Fit SARIMAX sur la série ou sur les résidus
    fit_sarimax = pm.auto_arima(
        y=resid,
        X=X_train,
        seasonal=False,       # si tu veux ajouter saisonnalité via STL
        suppress_warnings=True,
        error_action="ignore",
        stepwise=True
    )


    # tendance future
    t = np.arange(len(trend)).reshape(-1,1)
    model_trend = LinearRegression().fit(t, trend)
    n_trend = len(trend)

    # saisonnalité répétée
    season_pattern = seasonal.iloc[-saisonalite:].values
    
    # Prédiction si test fourni
    if type(test) != bool:
        n_test = len(test_y)
        pred_resid = fit_sarimax.predict(n_periods=n_test, X=test_X)
        
        # tendance future
        t_future = np.arange(n_trend, n_trend+n_test).reshape(-1,1)
        trend_future = model_trend.predict(t_future)

        # saisonnalité répétée
        season_future = np.tile(season_pattern, int(np.ceil(n_test/saisonalite)))[:n_test]
        
        # reconstruction
        y_pred = pred_resid + trend_future + season_future
        
        # retour à l'échelle originale
        if type_produit:
            y_pred = np.exp(y_pred) + min_train - 1
        
        rmse = np.sqrt(mean_squared_error(test_y, y_pred))
        mae = mean_absolute_error(test_y, y_pred)
        return fit_sarimax, rmse, mae, model_trend, n_trend, season_pattern
    
    return fit_sarimax, model_trend, n_trend, season_pattern

## LSTM

In [74]:
def create_sequences(data, taille_fenêtre):
    
    X = []
    y = []

    # on parcourt la série
    for i in range(len(data) - taille_fenêtre):
        
        # séquence d'entrée
        X.append(data[i:i+taille_fenêtre])
        
        # valeur à prédire
        y.append(data[i+taille_fenêtre])

    return np.array(X), np.array(y)

In [187]:
def LSTM(train, val, saisonalite, test=False, n_neurones = 32, optimizer="adam", loss="mse", metrics=["mae"], epochs=80, batch_size=16, patience=10):
    # test = False si on ne veut pas tester le modèle sur un ensmble de test

    # Le scaler transforme les données pour qu'elles soient entre 0 et 1.
    
    scaler = MinMaxScaler(feature_range=(0, 1))
    
    # On apprend le minimum et le maximum à partir du train seulement.
    # Cela évite d'utiliser des informations du futur.
    
    # Transformer en 2D pour sklearn
    train_scaled = scaler.fit_transform(train.values.reshape(-1,1))
    val_scaled   = scaler.transform(val.values.reshape(-1,1))
    
    if test is not None:
        test_scaled = scaler.transform(test.values.reshape(-1,1))

    # Création des séquences

    L = saisonalite

    # création des séquences pour train
    X_train, y_train = create_sequences(train_scaled, L)
    X_train = X_train.reshape((X_train.shape[0], X_train.shape[1], 1))
    X_val, y_val = create_sequences(val_scaled, L)
    X_val = X_val.reshape((X_val.shape[0], X_val.shape[1], 1))

    if type(test) != bool:
        # création des séquences pour test
        X_test, y_test = create_sequences(test_scaled, L)
        X_test = X_test.reshape((X_test.shape[0], X_test.shape[1], 1))

    # Les données sont transformées en séquences afin de permettre au LSTM
    # d'apprendre les dépendances temporelles.
    # 
    # On apprend au modèle : “Si les 12 derniers mois ressemblent à ceci → le mois suivant devrait être cela.”
    
    # Chaque entrée du modèle correspond à une fenêtre temporelle
    # contenant les L observations précédentes.
    #
    # La sortie correspond à la valeur suivante de la série.
    #
    # Cette transformation permet de reformuler le problème de prévision
    # en un problème supervisé adapté aux réseaux de neurones.

    # Construction du modèle LSTM

    # Création d'un modèle séquentiel
    model = Sequential()
    
    # Couche LSTM
    # 32 neurones = taille de la mémoire du réseau
    model = Sequential([
    Input(shape=(X_train.shape[1], 1)),
    lstm(n_neurones),
    Dense(1)
    ])

    model.compile(
    optimizer=optimizer,
    loss=loss,
    metrics=metrics
    )

    early_stop = EarlyStopping(
        monitor="val_loss",
        patience=patience,
        restore_best_weights=True
    )

    model.fit(
    X_train,
    y_train,
    epochs=epochs,  # nombre d’itérations d’apprentissage
    batch_size=batch_size,    # nombre d'exemples traités à la fois
    validation_data=(X_val,y_val),
    callbacks=[early_stop],
    shuffle=False # série temporelle
    )

    if type(test) != bool:
        # Évaluation du modèle
        predictions = model.predict(X_test)
    
        # Remettre les valeurs à l’échelle originale; Comme on a normalisé les données, il faut annuler la normalisation.
        predictions = scaler.inverse_transform(predictions)
        y_test_real = scaler.inverse_transform(y_test.reshape(-1,1))
    
        rmse_lstm = np.sqrt(mean_squared_error(y_test_real, predictions))
        mae_lstm = mean_absolute_error(y_test_real, predictions)
        
        return model, rmse_lstm, mae_lstm, scaler
    return model, scaler
    

## LSTM multi-pas direct

In [76]:
def create_sequences_multi(data, window_size, horizon):

    X = []
    y = []

    for i in range(len(data) - window_size - horizon + 1):

        X.append(data[i:i+window_size])
        y.append(data[i+window_size:i+window_size+horizon])

    return np.array(X), np.array(y)

In [193]:
def LSTM_multi_pas_direct(train, val, saisonalite, test=False, taille_fenetre=12, n_pas = 12, n_neurones = 32, optimizer="adam", 
                   loss="mse", metrics=["mae"], n_epochs=80, batch_size=16, patience=10):
    scaler = MinMaxScaler(feature_range=(0, 1))
    
    # On apprend le minimum et le maximum à partir du train seulement.
    # Cela évite d'utiliser des informations du futur.
    
    # Transformer en 2D pour sklearn
    train_scaled = scaler.fit_transform(train.values.reshape(-1,1))
    val_scaled   = scaler.transform(val.values.reshape(-1,1))


    if len(val_scaled) < taille_fenetre + n_pas:
        print('erreur, il faut un ensemble de validation plus grand !')
        return(None)

    # Création des séquences

    L = saisonalite
    X_train_multi, y_train_multi = create_sequences_multi(train_scaled, L, n_pas)
    X_train_multi = X_train_multi.reshape((X_train_multi.shape[0], X_train_multi.shape[1], 1))
    X_val_multi, y_val_multi = create_sequences_multi(val_scaled, L, n_pas)
    X_val_multi = X_val_multi.reshape((X_val_multi.shape[0], X_val_multi.shape[1], 1))

    if type(test) != bool:
        # On applique exactement la même transformation au jeu de test.
        test_scaled = scaler.transform(test.values.reshape(-1,1))
        if len(test_scaled) < taille_fenetre + n_pas:
            print('erreur, il faut un ensemble de test plus grand !')
            return(None)
        
        X_test_multi, y_test_multi = create_sequences_multi(test_scaled, L, n_pas)
        X_test_multi = X_test_multi.reshape((X_test_multi.shape[0], X_test_multi.shape[1], 1))

    # MODELE DIRECT MULTI-PAS

    model = Sequential([
    Input(shape=(taille_fenetre, 1)),
    lstm(n_neurones),
    Dense(n_pas) # multipas direct
    ])
    
    model.compile(
        optimizer=optimizer,
        loss=loss,
        metrics=metrics
    )

    early_stop = EarlyStopping(
        monitor="val_loss",
        patience=patience,
        restore_best_weights=True
    )

    
    model.fit(
        X_train_multi,
        y_train_multi,
        epochs=n_epochs,
        batch_size=batch_size,
        validation_data=(X_val_multi, y_val_multi),
        shuffle=False,
        callbacks=[early_stop]
    )

    if type(test) != bool:
        # Évaluation du modèle
        predictions = model.predict(X_test_multi)
    
        # Remettre les valeurs à l’échelle originale; Comme on a normalisé les données, il faut annuler la normalisation.
        predictions = scaler.inverse_transform(predictions.reshape(-1,1)).reshape(predictions.shape)
        y_test_real = scaler.inverse_transform(y_test_multi.reshape(-1,1)).reshape(y_test_multi.shape)
    
        rmse_lstm = np.sqrt(mean_squared_error(y_test_real.flatten(), predictions.flatten()))
        mae_lstm = mean_absolute_error(y_test_real.flatten(), predictions.flatten())
        
        return model, rmse_lstm, mae_lstm, scaler
    
    return model, scaler    

## ETS/Holt-Winters

In [316]:
def ETS(train, saisonalite, test=False, type_produit=False): # freq='MS' : données mensuelles au début du mois (Month Start)
    if test is not False:
        test = test.reindex(pd.date_range(start=test.index.min(), end=test.index.max(), freq='MS')).interpolate()

    
    # train doit avoir un index datetime
    if not isinstance(train.index, pd.DatetimeIndex):
        raise ValueError("train doit avoir un DatetimeIndex")

    # Transformation log si multiplicatif
    if type_produit:
        min_train = train.min()
        train = train - min_train + 1
        train = np.log(train)
    
    hw_model = ExponentialSmoothing(
        train,
        trend="add",
        seasonal="add",
        seasonal_periods=saisonalite
    ).fit()

    if type_produit:
        train = np.exp(train) + min_train - 1 # on remet le train comme avant

    if type(test) != bool:
        # Évaluation du modèle
        hw_pred = hw_model.forecast(len(test))

        # retour à l'échelle originale si log
        if type_produit:
            hw_pred = np.exp(hw_pred) + min_train - 1

        rmse_hw = np.sqrt(mean_squared_error(test, hw_pred))
        mae_hw = mean_absolute_error(test, hw_pred)
        
        return hw_model, rmse_hw, mae_hw
    
    return hw_model

# Exemple d'utilisation

In [307]:
# Exemple : dataset tourisme (fictive local)

dataset = load_dataset("zaai-ai/time_series_dataset_residuals")
# Choisir le split 'train' (ou 'test')
train_split = dataset['train']

# Convertir ce split en DataFrame
df = train_split.to_pandas()


# Target
y = df['residuals'][:1000]

# on normalise y
y = (y - y.mean()) / y.std() # on augmente artificiellement l'écart-type de la série standardisée pour garantir un fonctionnement des ARIMA

# proportions
train_frac = 0.6
val_frac = 0.2
test_frac = 0.2

n_total = len(y)
# print(n_total) # DEBUG

# indices de split
train_end = int(n_total * train_frac)
val_end   = int(n_total * (train_frac + val_frac))

# Garde un DatetimeIndex
df['Date'] = pd.to_datetime(df['Date'])
df.set_index('Date', inplace=True)

# Splits
train = y.iloc[:train_end]
val   = y.iloc[train_end:val_end]
test  = y.iloc[val_end:]

# Réassignation de l'index datetime
train.index = df.index[:train_end]
val.index   = df.index[train_end:val_end]
test.index  = df.index[val_end:val_end + len(test)]

train_val = pd.concat([train, val])
train_val.index = df.index[:val_end]  # <- DatetimeIndex correct

## Persistence

In [277]:
prediction, rmse, mae = persistence(train_val, test)

In [278]:
print(rmse, mae)
prediction * np.ones(10)

0.6794419482230164 0.3679369964140396


array([-0.21154174, -0.21154174, -0.21154174, -0.21154174, -0.21154174,
       -0.21154174, -0.21154174, -0.21154174, -0.21154174, -0.21154174])

## ARIMA

In [279]:
fit_arima, rmse_arima, mae_arima = ARIMA(train_val, 12, test=test, type_produit=True) # 12 mois par an

In [280]:
print(rmse_arima, mae_arima)
predictions = np.exp(fit_arima.predict(n_periods = len(test))) + train.min() - 1
predictions[:10]

0.6578980320844144 0.37603503754163015


800    1.173037
801    1.173037
802    1.173037
803    1.173037
804    1.173037
805    1.173037
806    1.173037
807    1.173037
808    1.173037
809    1.173037
dtype: float64

## STL + ARIMA

In [281]:
model_trend, season_pattern, fit_resid, rmse_stl_arima, mae_stl_arima = STL_ARIMA(train_val, 12, 13, test=test, type_produit=True)

In [282]:
print(rmse_stl_arima, mae_stl_arima)

4.951386078283856 4.906660754029423


In [283]:
def forecast_stl_arima(model_trend, season_pattern, min_train, fit_resid, n_train, n_test, saisonalite, type_produit=False):

    # Prévision des résidus
    pred_resid = fit_resid.predict(n_periods=n_test)

    # tendance
    t_future = np.arange(n_train, n_train+n_test).reshape(-1,1)
    trend_future = model_trend.predict(t_future)

    # répétition de la saisonnalité
    season_future = np.tile(season_pattern, int(np.ceil(n_test/saisonalite)))[:n_test] # tile répète le motif saisonnier

    # reconstruction
    y_pred = trend_future + season_future + pred_resid

    # Retour échelle originale
    if type_produit:
        y_pred = np.exp(y_pred) + min_train - 1 # on remet le train comme avant

    return y_pred

In [284]:
predictions = forecast_stl_arima(model_trend, season_pattern, train.min(), fit_resid, len(train_val), len(test), 12, type_produit=True)
predictions[:10]

800    1.163752
801    1.189086
802    1.247950
803    1.194115
804    1.163125
805    1.208956
806    1.240331
807    1.185567
808    1.201687
809    1.275773
dtype: float64

## SARIMAX

### Nouvelles données (besoin de covariables)

In [285]:
# Covariables dynamiques
X = df[['CPI', 'Inflation_Rate', 'GDP']][:1000]

# On standardise
X = (X - X.mean()) / X.std()

# Train
X_train = X.iloc[:train_end].astype(float)

# Validation
X_val = X.iloc[train_end:val_end].astype(float)

# Test
X_test = X.iloc[val_end:].astype(float)

# Train-Val
X_train_val = pd.concat([X_train, X_val])

# on standardise y
y = (y - y.mean()) / y.std()

# création des ensembles
train = y.iloc[:train_end]
val   = y.iloc[train_end:val_end]
test  = y.iloc[val_end:]

train_val = pd.concat([train, val])

In [286]:
# entraînement + évaluation
modele, rmse, mae = SARIMAX_model(
    train_val,
    X_train_val,
    saisonalite=12,
    test_y = test,
    test_X = X_test,
    type_produit=True
)

print("RMSE :", rmse)
print("MAE :", mae)

RMSE : 4.924806557209839
MAE : 4.879540238244414


In [287]:
predictions = modele.predict(n_periods=len(test),        # nombre de périodes futures à prédire.
            X=X_test)      # Comme le modèle a été entraîné avec des variables exogènes, il a besoin des valeurs de ces variables sur la période future
y_pred = np.exp(predictions) + train_val.min() - 1
y_pred[:10]

800   -0.079195
801   -0.079195
802   -0.049067
803   -0.049067
804   -0.049067
805   -0.049067
806   -0.049067
807   -0.049067
808   -0.049067
809   -0.049067
dtype: float64

## STL + ARIMAX

In [288]:
# entraînement + évaluation
modele, rmse, mae, model_trend, n_trend, season_pattern = SARIMAX_STL_model(train, X_train, 12, 13, test_y=test, test_X=X_test, type_produit=True)

print("RMSE :", rmse)
print("MAE :", mae)

RMSE : 2.6838497536822175
MAE : 2.473492405319175


In [289]:
pred_resid = modele.predict(n_periods=len(test), X=X_test)
n_test = len(test)

# tendance future
t_future = np.arange(n_trend, n_trend+n_test).reshape(-1,1)
trend_future = model_trend.predict(t_future)

# saisonnalité répétée
season_future = np.tile(season_pattern, int(np.ceil(n_test/12)))[:n_test]

# reconstruction
y_pred = pred_resid + trend_future + season_future

# retour à l'échelle originale
y_pred = np.exp(y_pred) + train.min() - 1
y_pred[:10]

600   -2.117014
601   -2.026178
602   -3.380966
603   -3.336842
604   -3.136910
605    1.583766
606   -2.363937
607   -2.542530
608   -0.030236
609   -2.943352
dtype: float64

## LSTM

### Simple

In [290]:
fit_LSTM, rmse_LSTM, mae_LSTM, scaler_simple = LSTM(train, val, 12, test=test) # 12 mois par an

Epoch 1/80
37/37 ━━━━━━━━━━━━━━━━━━━━ 2s 15ms/step - loss: 0.0260 - mae: 0.1188 - val_loss: 0.0087 - val_mae: 0.0468
Epoch 2/80
37/37 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 0.0135 - mae: 0.0764 - val_loss: 0.0087 - val_mae: 0.0480
Epoch 3/80
37/37 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 0.0133 - mae: 0.0763 - val_loss: 0.0087 - val_mae: 0.0459
Epoch 4/80
37/37 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 0.0133 - mae: 0.0764 - val_loss: 0.0087 - val_mae: 0.0450
Epoch 5/80
37/37 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 0.0133 - mae: 0.0766 - val_loss: 0.0087 - val_mae: 0.0447
Epoch 6/80
37/37 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 0.0133 - mae: 0.0767 - val_loss: 0.0087 - val_mae: 0.0446
Epoch 7/80
37/37 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 0.0133 - mae: 0.0768 - val_loss: 0.0087 - val_mae: 0.0446
Epoch 8/80
37/37 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 0.0133 - mae: 0.0768 - val_loss: 0.0087 - val_mae: 0.0446
Epoch 9/80
37/37 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 0.0133 - mae:

In [291]:
print(rmse_LSTM, mae_LSTM)

0.6068051523949685 0.38084282548419074


### Multi-pas direct

In [292]:
fit_LSTM_direct, rmse_LSTM_direct, mae_LSTM_direct, scaler_multi = LSTM_multi_pas_direct(train, val, 12, test=test) 

Epoch 1/80
37/37 ━━━━━━━━━━━━━━━━━━━━ 3s 16ms/step - loss: 0.0362 - mae: 0.1470 - val_loss: 0.0102 - val_mae: 0.0555
Epoch 2/80
37/37 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 0.0138 - mae: 0.0791 - val_loss: 0.0092 - val_mae: 0.0451
Epoch 3/80
37/37 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 0.0133 - mae: 0.0777 - val_loss: 0.0091 - val_mae: 0.0454
Epoch 4/80
37/37 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 0.0133 - mae: 0.0776 - val_loss: 0.0091 - val_mae: 0.0456
Epoch 5/80
37/37 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 0.0133 - mae: 0.0777 - val_loss: 0.0091 - val_mae: 0.0456
Epoch 6/80
37/37 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 0.0133 - mae: 0.0777 - val_loss: 0.0091 - val_mae: 0.0456
Epoch 7/80
37/37 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 0.0133 - mae: 0.0777 - val_loss: 0.0091 - val_mae: 0.0456
Epoch 8/80
37/37 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 0.0133 - mae: 0.0777 - val_loss: 0.0091 - val_mae: 0.0455
Epoch 9/80
37/37 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 0.0133 - mae:

In [293]:
print(rmse_LSTM_direct, mae_LSTM_direct)

0.6085422985978199 0.3872839468220165


### Prévision multi-pas LSTM récursif

In [294]:
def forecast_lstm(model, val, scaler, saisonalite, n_steps):

    # scaler comme dans l'entraînement
    val_scaled = scaler.transform(val.values.reshape(-1, 1))

    # dernière fenêtre
    last_window = val_scaled[-saisonalite:]

    preds = []

    for _ in range(n_steps):

        X = last_window.reshape(1, saisonalite, 1)

        pred = model.predict(X, verbose=0)

        preds.append(pred[0,0])

        # mise à jour de la fenêtre
        last_window = np.vstack([last_window[1:], pred])

    preds = np.array(preds).reshape(-1,1)

    # retour à l'échelle originale
    preds = scaler.inverse_transform(preds)

    return preds

In [295]:
predictions = forecast_lstm(
    fit_LSTM,
    val,
    scaler_simple,
    saisonalite=12,
    n_steps=10 # prédit les 10 prochaines périodes
)
predictions

array([[-0.09746167],
       [-0.09334274],
       [-0.08976664],
       [-0.08426759],
       [-0.0792254 ],
       [-0.07432854],
       [-0.06935815],
       [-0.06715725],
       [-0.06399661],
       [-0.05995788]], dtype=float32)

### Prévision multi-pas LSTM direct

In [296]:
def forecast_lstm_direct(model, val, scaler, saisonalite):
    # scaler comme dans ton entraînement
    val_scaled = scaler.transform(val.values.reshape(-1, 1))

    # dernière fenêtre
    last_window = val_scaled[-saisonalite:]

    # Mise en forme pour LSTM
    X = last_window.reshape(1, saisonalite, 1)

    # Prédiction directe
    preds = model.predict(X)

    # Retour à l’échelle originale
    preds = scaler.inverse_transform(preds.reshape(-1,1)).reshape(preds.shape)

    return preds.flatten()

In [297]:
predictions = forecast_lstm_direct(
    fit_LSTM_direct,   # modèle direct multi-step
    val,
    scaler_multi,
    saisonalite=12
)
predictions

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 53ms/step


array([-0.05241283, -0.04021411, -0.07445564, -0.08191268, -0.03467568,
       -0.05645735, -0.04351834, -0.07932384, -0.07154833, -0.08123811,
       -0.08546329, -0.10489381], dtype=float32)

## ETS/Holt-Winter

In [347]:
# Exemple : dataset tourisme (fictive local)

dataset = load_dataset("zaai-ai/time_series_dataset_residuals")
# Choisir le split 'train' (ou 'test')
train_split = dataset['train']

# Convertir ce split en DataFrame
df = train_split.to_pandas()


# Target
y = df['residuals'][:1000]

# on normalise y
y = (y - y.mean()) / y.std() # on augmente artificiellement l'écart-type de la série standardisée pour garantir un fonctionnement des ARIMA

# proportions
train_frac = 0.6
val_frac = 0.2
test_frac = 0.2

n_total = len(y)
# print(n_total) # DEBUG

# indices de split
train_end = int(n_total * train_frac)
val_end   = int(n_total * (train_frac + val_frac))

# Garde un DatetimeIndex
df['Date'] = pd.to_datetime(df['Date'])
df.set_index('Date', inplace=True)

# Splits
train = y.iloc[:train_end]
val   = y.iloc[train_end:val_end]
test  = y.iloc[val_end:]

# Réassignation de l'index datetime
train.index = df.index[:train_end]
val.index   = df.index[train_end:val_end]
test.index  = df.index[val_end:val_end + len(test)]

train_val = pd.concat([train, val])
train_val.index = df.index[:val_end]  # <- DatetimeIndex correct

In [348]:
# supprimer les doublons (si présents)
# Vérifie les doublons
print(train_val.index.duplicated().sum())
print(test.index.duplicated().sum())
train_val = train_val[~train_val.index.duplicated(keep='first')]
test = test[~test.index.duplicated(keep='first')]

630
30


In [349]:
print(test.index.is_monotonic_increasing)
print(test.index.is_monotonic_increasing)
# on trie par date et on réindexe en attribuant une fréquence

train_val = train_val.sort_index()
train_val = train_val.reindex(pd.date_range(start=test.index.min(), end=test.index.max(), freq='MS')).interpolate()

test = test.sort_index()
test = test.reindex(pd.date_range(start=test.index.min(), end=test.index.max(), freq='MS')).interpolate()

False
False


In [350]:
hw_model, rmse_hw, mae_hw = ETS(train_val, 12, type_produit=True, test=test)

In [351]:
print(rmse_hw, mae_hw)
predictions = np.exp(hw_model.forecast(len(test))) + train.min() - 1
predictions[:10]

0.9022904109815684 0.7550638231604518


2017-01-01   -1.312431
2017-02-01   -0.069861
2017-03-01    0.026536
2017-04-01   -0.678121
2017-05-01   -0.041258
2017-06-01   -0.502365
2017-07-01   -0.107730
2017-08-01   -0.292174
2017-09-01   -0.406536
2017-10-01    0.117646
Freq: MS, dtype: float64